# 05 Baseline Modeling

This notebook trains the main candidate models used in the project and stores their predictions for later evaluation.

In [11]:
from sklearn.metrics import classification_report

In [12]:
# Author: Jessica
# Loading model-ready data so this notebook runs independently

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

inspection_model_df = pd.read_csv(
    "shared_data/inspection_model_df.csv",
    parse_dates=['INSPECTION DATE', 'next_inspection_date']
)
print(inspection_model_df.shape)
inspection_model_df.head()

(48424, 62)


,CAMIS,INSPECTION DATE,score,grade,dba_clean,boro,cuisine_description,inspection_type,street_num,street_name,...,lat_bin,lon_bin,grid_cell,zip_avg_score,zip_restaurant_count,median_household_income,population_density_per_sq_mi,poverty_rate_pct,zip_total_inspections,zip_inspections_per_restaurant
0,30075445,2023-08-01,38.0,NaN,MORRIS PARK BAKE SHOP,Bronx,Bakery Products/Desserts,Cycle Inspection / Initial Inspection,1007.0,MORRIS PARK AVENUE,...,21.0,17.0,21.0_17.0,18.759197,129.0,40935.0,34653.0,27.3,304.0,2.356589
1,30075445,2023-08-22,12.0,A,MORRIS PARK BAKE SHOP,Bronx,Bakery Products/Desserts,Cycle Inspection / Re-inspection,1007.0,MORRIS PARK AVENUE,...,21.0,17.0,21.0_17.0,18.759197,129.0,40935.0,34653.0,27.3,304.0,2.356589
2,30191841,2023-04-23,10.0,A,D.J. REYNOLDS,Manhattan,Irish,Cycle Inspection / Initial Inspection,351.0,WEST 57 STREET,...,16.0,12.0,16.0_12.0,18.794383,486.0,93651.0,72918.0,14.3,1002.0,2.061728
3,30191841,2024-11-20,24.0,NaN,D.J. REYNOLDS,Manhattan,Irish,Cycle Inspection / Initial Inspection,351.0,WEST 57 STREET,...,16.0,12.0,16.0_12.0,18.794383,486.0,93651.0,72918.0,14.3,1002.0,2.061728
4,40356018,2024-04-16,0.0,A,RIVIERA CATERERS,Brooklyn,American,Administrative Miscellaneous / Initial Inspection,2780.0,STILLWELL AVENUE,...,4.0,12.0,4.0_12.0,17.535354,54.0,67940.0,39438.0,19.2,100.0,1.851852


## Scope



In [13]:
# Author: Jessica
# Edited by: Zeek & Mian — added geo/socioeconomic features from notebook 04
# Setting up features and target for baseline model predicting inspection likelihood

features = [
    'historical_avg_score',
    'historical_avg_critical',
    'prior_inspection_count',
    'critical_violations',
    'total_violations',
    'inspection_month',
    'days_since_last_inspection',
    'score_change_from_previous',
    # Joel: time since open proxy
    'time_since_open_days',
    # Zeek: raw geo
    'latitude',
    'longitude',
    'zipcode',
    # Zeek: zipcode-level proxies
    'zip_avg_score',
    'zip_restaurant_count',
    'zip_inspections_per_restaurant',
    # Zeek: borough-level socioeconomic
    'median_household_income',
    'population_density_per_sq_mi',
    'poverty_rate_pct',
    # Zeek: boro one-hot columns
    'boro_Bronx',
    'boro_Brooklyn',
    'boro_Manhattan',
    'boro_Queens',
    'boro_Staten Island'
]

# Only use features that exist in the dataframe
features = [f for f in features if f in inspection_model_df.columns]

print('Features used:', features)
print('Model matrix shape:', inspection_model_df[features].shape)

Features used: ['historical_avg_score', 'historical_avg_critical', 'prior_inspection_count', 'critical_violations', 'total_violations', 'inspection_month', 'days_since_last_inspection', 'score_change_from_previous', 'time_since_open_days', 'latitude', 'longitude', 'zipcode', 'zip_avg_score', 'zip_restaurant_count', 'zip_inspections_per_restaurant', 'median_household_income', 'population_density_per_sq_mi', 'poverty_rate_pct', 'boro_Bronx', 'boro_Brooklyn', 'boro_Manhattan', 'boro_Queens', 'boro_Staten Island']
Model matrix shape: (48424, 23)


In [14]:
# Author: Jessica
# Taking the union of all restaurants so the model has seen every restaurant
# in the test set during training. Restaurants with no training history get
# a placeholder row with zeroed features. This avoids the blind spot of
# never-inspected restaurants being completely invisible to the model.

from sklearn.model_selection import train_test_split

X = inspection_model_df[features].fillna(0)
y = inspection_model_df['inspection_within_90_days']

X_with_camis = X.copy()
X_with_camis['CAMIS'] = inspection_model_df['CAMIS'].values

# Split as usual
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_with_camis, y, test_size=0.2, random_state=42, stratify=y
)

# Find restaurants in test with no rows in train
train_camis = set(X_train_full['CAMIS'])
test_camis = set(X_test_full['CAMIS'])
unseen_camis = test_camis - train_camis

print(f"Total unique restaurants in test set: {len(test_camis)}")
print(f"Restaurants unseen during training: {len(unseen_camis)}")

# Add zeroed placeholder row for each unseen restaurant
if unseen_camis:
    placeholder_X = pd.DataFrame(
        np.zeros((len(unseen_camis), X_train_full.shape[1])),
        columns=X_train_full.columns
    )
    placeholder_X['CAMIS'] = list(unseen_camis)
    placeholder_y = pd.Series(np.zeros(len(unseen_camis), dtype=int))

    X_train_full = pd.concat([X_train_full, placeholder_X], ignore_index=True)
    y_train = pd.concat([y_train.reset_index(drop=True), placeholder_y], ignore_index=True)

print(f"Training set size after union: {X_train_full.shape[0]}")

# Drop CAMIS before passing to model
X_train = X_train_full.drop(columns=['CAMIS'])
X_test = X_test_full.drop(columns=['CAMIS'])

Total unique restaurants in test set: 7984
Restaurants unseen during training: 1672
Training set size after union: 40411


In [15]:
# Author: Jessica
# Adding StandardScaler so logistic regression converges properly
# with the new geo and socioeconomic features

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000))
])
lr_pipeline.fit(X_train, y_train)
y_pred = lr_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.95      0.88      7548
           1       0.60      0.26      0.36      2137

    accuracy                           0.80      9685
   macro avg       0.71      0.61      0.62      9685
weighted avg       0.77      0.80      0.77      9685



In [16]:
# Author: Jessica
# Balanced logistic regression with scaling

lr_balanced_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, class_weight='balanced'))
])
lr_balanced_pipeline.fit(X_train, y_train)
y_pred_balanced = lr_balanced_pipeline.predict(X_test)
print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.90      0.74      0.81      7548
           1       0.44      0.70      0.54      2137

    accuracy                           0.73      9685
   macro avg       0.67      0.72      0.67      9685
weighted avg       0.79      0.73      0.75      9685



In [17]:
# Author: Jessica
# Training a Random Forest model to compare performance with logistic regression

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.84      0.93      0.89      7548
           1       0.62      0.38      0.47      2137

    accuracy                           0.81      9685
   macro avg       0.73      0.66      0.68      9685
weighted avg       0.79      0.81      0.79      9685



Tuning improved precision and accuracy but did not improve recall, so balanced logistic regression remains the better model for identifying inspections

## Saved Output

This notebook saves model results and predictions for use in the evaluation notebook.

In [18]:
# Author: Jessica
# Saving model predictions and coefficients so the evaluation notebook can run independently
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

predictions_df = pd.DataFrame({
    'y_test': y_test.reset_index(drop=True),
    'logistic_pred': y_pred,
    'balanced_logistic_pred': y_pred_balanced,
    'random_forest_pred': y_pred_rf
})
predictions_df.to_csv("shared_data/model_predictions.csv", index=False)

model_results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Balanced Logistic Regression', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_balanced),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Precision_Inspection': [
        precision_score(y_test, y_pred),
        precision_score(y_test, y_pred_balanced),
        precision_score(y_test, y_pred_rf)
    ],
    'Recall_Inspection': [
        recall_score(y_test, y_pred),
        recall_score(y_test, y_pred_balanced),
        recall_score(y_test, y_pred_rf)
    ],
    'F1_Inspection': [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_balanced),
        f1_score(y_test, y_pred_rf)
    ]
})
model_results.to_csv("shared_data/model_results_summary.csv", index=False)

importance = pd.DataFrame({
    'feature': features,
    'coefficient': lr_balanced_pipeline.named_steps['model'].coef_[0]
}).sort_values(by='coefficient', ascending=False)
importance.to_csv("shared_data/feature_importance.csv", index=False)

print("Saved model_predictions.csv, model_results_summary.csv, and feature_importance.csv")

Saved model_predictions.csv, model_results_summary.csv, and feature_importance.csv
